In [ ]:
!pip install vertexai

In [6]:
import json
import os
import urllib.parse
from google.cloud import storage
import vertexai
from vertexai.generative_models import GenerativeModel, Part, HarmCategory, HarmBlockThreshold

project_id = "segfault-434120"
region = "us-central1"
bucket_name = "skincare-products"
in_folder = "raw/skin_disease_pictures/"
out_folder = "raw/skin_disease_pictures/raw/"
model_name = "gemini-1.5-flash-001"

prompt_template = """The current file name is '{filename}'.
1. Identify the whole disease name from the file name and use underscores.
2. Interpret the skin color from the image.
3. Identify the severity level (mild, moderate, severe).

Return the output as JSON with the schema: skin_condition_name:string, severity:string, skin_color:string.
Do not include any other fields."""

def main():
    vertexai.init(project=project_id, location=region)
    model = GenerativeModel(model_name)
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)

    if not os.path.exists(out_folder):
        os.makedirs(out_folder)

    disease_indices = {}

    blobs = storage_client.list_blobs(bucket_name, prefix=in_folder)

    for blob in blobs:
        if blob.name.startswith(out_folder) or not blob.name.endswith(".jpg"):
            continue

        image_filename = os.path.basename(blob.name)

        encoded_blob_name = urllib.parse.quote(blob.name)

        image_link = f"https://storage.cloud.google.com/{bucket_name}/{encoded_blob_name}"

        print(f"Processing image: {image_link} with filename: {image_filename}")

        prompt = prompt_template.format(filename=image_filename)

        image_path = f"gs://{bucket_name}/{blob.name}"
        image_file = Part.from_uri(image_path, "image/jpeg")

        try:
            resp = model.generate_content(
                contents=[image_file, prompt],
                safety_settings={
                    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
                    HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_ONLY_HIGH,
                    HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
                    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
                }
            )
            resp_str = resp.text.replace("```json", "").replace("```", "")
            formatted_resp_str = resp_str.replace("\n", "").replace("\r", "").strip()
            parsed_response = json.loads(formatted_resp_str)

            # extract the skin condition name, severity, and skin color
            disease_name = parsed_response.get("skin_condition_name", "").lower()
            severity = parsed_response.get("severity", "")
            skin_color = parsed_response.get("skin_color", "")

            # update the index for each disease name
            if disease_name not in disease_indices:
                disease_indices[disease_name] = 1
            else:
                disease_indices[disease_name] += 1

            # filename in the format: disease_name-index
            standardized_filename = f"{disease_name}-{disease_indices[disease_name]}"

            data = {
                "skin_condition_name": disease_name,
                "skin_color": skin_color,
                "severity": severity,
                "image_link": image_link
            }
            json_filename = f"{standardized_filename}.json"
            json_blob = bucket.blob(f"{out_folder}{json_filename}")
            json_blob.upload_from_string(json.dumps(data, separators=(',', ':'), ensure_ascii=False), content_type='application/json')

            print(f"Uploaded JSON file for {image_filename} to GCS as {json_filename}")

        except Exception as e:
            print(f"Skipping due to safety block for {blob.name}: {e}")

if __name__ == "__main__":
    main()


Streaming output truncated to the last 5000 lines.
Processing image: https://storage.cloud.google.com/skincare-products/raw/skin_disease_pictures/Nail%20Fungus%20and%20other%20Nail%20Disease/pseudomonas-13.jpg with filename: pseudomonas-13.jpg
Uploaded JSON file for pseudomonas-13.jpg to GCS as pseudomonas-2.json
Processing image: https://storage.cloud.google.com/skincare-products/raw/skin_disease_pictures/Nail%20Fungus%20and%20other%20Nail%20Disease/pseudomonas-15.jpg with filename: pseudomonas-15.jpg
Uploaded JSON file for pseudomonas-15.jpg to GCS as pseudomonas-3.json
Processing image: https://storage.cloud.google.com/skincare-products/raw/skin_disease_pictures/Nail%20Fungus%20and%20other%20Nail%20Disease/pseudomonas-18.jpg with filename: pseudomonas-18.jpg
Uploaded JSON file for pseudomonas-18.jpg to GCS as pseudomonas-4.json
Processing image: https://storage.cloud.google.com/skincare-products/raw/skin_disease_pictures/Nail%20Fungus%20and%20other%20Nail%20Disease/pseudomonas-19.jp